# Environment and GPU hardware check

Run this notebook on the remote GPU/CUDA server before training the project notebooks. It collects the practical facts that usually matter for this repository: Python version, installed package versions, CUDA visibility, GPU model and memory, PyTorch CUDA support, PyTorch Geometric health, RDKit health, and a small GPU smoke test.

The notebook does not train models. If `SAVE_REPORT=True`, it writes a timestamped JSON report to `data/environment_reports/` so you can compare local vs. cluster environments later.

In [1]:
from __future__ import annotations

import importlib
import importlib.metadata as importlib_metadata
import json
import os
import platform
import shutil
import site
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = DATA_DIR / "environment_reports"
SAVE_REPORT = True

report = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
}

def run_command(cmd, timeout=30):
    """Run a command safely and return a serializable result."""
    try:
        completed = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=timeout,
            check=False,
        )
        return {
            "cmd": cmd,
            "returncode": completed.returncode,
            "stdout": completed.stdout.strip(),
            "stderr": completed.stderr.strip(),
        }
    except FileNotFoundError as exc:
        return {"cmd": cmd, "error": f"not found: {exc}"}
    except subprocess.TimeoutExpired as exc:
        return {"cmd": cmd, "error": f"timeout after {timeout}s", "stdout": exc.stdout, "stderr": exc.stderr}

print(f"Project root: {PROJECT_ROOT}")
print(f"Report timestamp UTC: {report['created_utc']}")

Project root: /workspace
Report timestamp UTC: 2026-06-25T14:23:29.090144+00:00


## 1. Python, OS, and runtime basics

In [2]:
python_info = {
    "python_version": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "hostname": platform.node(),
    "cwd": str(Path.cwd()),
    "site_packages": site.getsitepackages() if hasattr(site, "getsitepackages") else [],
    "user_site": site.getusersitepackages(),
}
report["python"] = python_info

for key, value in python_info.items():
    print(f"{key}: {value}")

python_version: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
python_executable: /app/.venv/bin/python
platform: Linux-4.18.0-553.100.1.el8_10.x86_64-x86_64-with-glibc2.35
machine: x86_64
processor: x86_64
hostname: gpu0203
cwd: /workspace
site_packages: ['/app/.venv/lib/python3.10/site-packages', '/app/.venv/local/lib/python3.10/dist-packages', '/app/.venv/lib/python3/dist-packages', '/app/.venv/lib/python3.10/dist-packages']
user_site: /home/mziegle/.local/lib/python3.10/site-packages


## 2. Important environment variables

These are especially useful on SLURM/PBS clusters, where the scheduler may decide which GPUs are visible to your job.

In [3]:
interesting_env = [
    "CUDA_VISIBLE_DEVICES",
    "NVIDIA_VISIBLE_DEVICES",
    "CONDA_DEFAULT_ENV",
    "VIRTUAL_ENV",
    "PATH",
    "LD_LIBRARY_PATH",
    "SLURM_JOB_ID",
    "SLURM_JOB_NAME",
    "SLURM_NODELIST",
    "SLURM_GPUS",
    "SLURM_GPUS_ON_NODE",
    "SLURM_CPUS_PER_TASK",
    "PBS_JOBID",
    "PBS_NODEFILE",
]
env_info = {key: os.environ.get(key) for key in interesting_env if os.environ.get(key) is not None}
report["environment_variables"] = env_info

for key in interesting_env:
    print(f"{key}: {os.environ.get(key, '<unset>')}")

CUDA_VISIBLE_DEVICES: 0
NVIDIA_VISIBLE_DEVICES: all
CONDA_DEFAULT_ENV: <unset>
VIRTUAL_ENV: <unset>
PATH: /usr/local/bin:/app/.venv/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/home/mziegle/.local/bin:/home/mziegle/bin
LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64:/.singularity.d/libs
SLURM_JOB_ID: 349373
SLURM_JOB_NAME: jupyter-course
SLURM_NODELIST: gpu0203
SLURM_GPUS: <unset>
SLURM_GPUS_ON_NODE: 1
SLURM_CPUS_PER_TASK: 4
PBS_JOBID: <unset>
PBS_NODEFILE: <unset>


## 3. Required package versions

This reads `requirements.txt` from the project root and checks whether each top-level package can be resolved in the active Python environment.

In [4]:
requirements_path = PROJECT_ROOT / "requirements.txt"
required = []
if requirements_path.exists():
    for line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            required.append(line)
else:
    print("No requirements.txt found in project root.")

def distribution_name(requirement_line: str) -> str:
    for sep in ["==", ">=", "<=", "~=", ">", "<", "["]:
        if sep in requirement_line:
            return requirement_line.split(sep, 1)[0].strip()
    return requirement_line.strip()

package_info = {}
for req in required:
    name = distribution_name(req)
    try:
        version = importlib_metadata.version(name)
        package_info[name] = {"required_line": req, "installed": True, "version": version}
    except importlib_metadata.PackageNotFoundError:
        package_info[name] = {"required_line": req, "installed": False, "version": None}

report["requirements"] = package_info

for name, info in package_info.items():
    status = info["version"] if info["installed"] else "MISSING"
    print(f"{name:20s} {status}")

torch                2.10.0+cu126
torch_geometric      MISSING
ipywidgets           8.1.8
ipykernel            7.2.0
matplotlib           3.10.8
numpy                2.2.6
scipy                1.15.3
networkx             3.4.2
rdkit                MISSING
cython               MISSING
tqdm                 4.67.3
pandas               2.3.3
pyarrow              23.0.1


## 4. CUDA tools visible on the system

`nvidia-smi` checks the NVIDIA driver/runtime view. `nvcc --version` checks whether the CUDA compiler toolkit is installed. PyTorch can use CUDA even when `nvcc` is absent, but building some custom extensions may require it.

In [5]:
tool_paths = {tool: shutil.which(tool) for tool in ["nvidia-smi", "nvcc", "gcc", "g++"]}
report["tool_paths"] = tool_paths
for tool, path in tool_paths.items():
    print(f"{tool:12s}: {path or '<not found>'}")

cuda_commands = {
    "nvidia_smi_summary": run_command([
        "nvidia-smi",
        "--query-gpu=index,name,driver_version,memory.total,memory.free,compute_cap,pci.bus_id",
        "--format=csv,noheader",
    ]),
    "nvidia_smi_full": run_command(["nvidia-smi"]),
    "nvcc_version": run_command(["nvcc", "--version"]),
}
report["cuda_commands"] = cuda_commands

print("\n--- nvidia-smi GPU summary ---")
print(cuda_commands["nvidia_smi_summary"].get("stdout") or cuda_commands["nvidia_smi_summary"].get("error") or cuda_commands["nvidia_smi_summary"].get("stderr"))
print("\n--- nvcc --version ---")
print(cuda_commands["nvcc_version"].get("stdout") or cuda_commands["nvcc_version"].get("error") or cuda_commands["nvcc_version"].get("stderr"))

nvidia-smi  : /usr/bin/nvidia-smi
nvcc        : /usr/local/cuda/bin/nvcc
gcc         : /usr/bin/gcc
g++         : /usr/bin/g++

--- nvidia-smi GPU summary ---
0, NVIDIA A100-SXM4-80GB, 590.48.01, 81920 MiB, 81153 MiB, 8.0, 00000000:83:00.0

--- nvcc --version ---
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Tue_Oct_29_23:50:19_PDT_2024
Cuda compilation tools, release 12.6, V12.6.85
Build cuda_12.6.r12.6/compiler.35059454_0


## 5. PyTorch CUDA diagnostics

In [6]:
torch_info = {"import_ok": False}

try:
    import torch

    torch_info.update({
        "import_ok": True,
        "torch_version": torch.__version__,
        "torch_cuda_build": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_count": torch.cuda.device_count(),
        "cudnn_available": torch.backends.cudnn.is_available(),
        "cudnn_version": torch.backends.cudnn.version(),
    })

    devices = []
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        devices.append({
            "index": idx,
            "name": props.name,
            "total_memory_gb": round(props.total_memory / 1024**3, 3),
            "major": props.major,
            "minor": props.minor,
            "multi_processor_count": props.multi_processor_count,
        })
    torch_info["devices"] = devices

except Exception as exc:
    torch_info["error"] = repr(exc)

report["torch"] = torch_info
print(json.dumps(torch_info, indent=2))

{
  "import_ok": true,
  "torch_version": "2.10.0+cu126",
  "torch_cuda_build": "12.6",
  "cuda_available": true,
  "cuda_device_count": 1,
  "cudnn_available": true,
  "cudnn_version": 91002,
  "devices": [
    {
      "index": 0,
      "name": "NVIDIA A100-SXM4-80GB",
      "total_memory_gb": 79.251,
      "major": 8,
      "minor": 0,
      "multi_processor_count": 108
    }
  ]
}


## 6. GPU smoke test

This allocates a moderate tensor on the first CUDA device and performs matrix multiplications. If this cell succeeds, PyTorch can actually execute kernels on the GPU from this notebook kernel.

In [7]:
gpu_smoke = {"ran": False}

try:
    import torch
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
        torch.cuda.set_device(device)
        torch.cuda.empty_cache()

        n = 4096
        repeats = 8
        dtype = torch.float16 if torch.cuda.get_device_capability(device)[0] >= 7 else torch.float32
        a = torch.randn((n, n), device=device, dtype=dtype)
        b = torch.randn((n, n), device=device, dtype=dtype)

        torch.cuda.synchronize()
        start = time.perf_counter()
        c = None
        for _ in range(repeats):
            c = a @ b
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        gpu_smoke = {
            "ran": True,
            "device": torch.cuda.get_device_name(device),
            "dtype": str(dtype),
            "matrix_size": n,
            "repeats": repeats,
            "elapsed_seconds": elapsed,
            "approx_tflops": round((2 * n**3 * repeats) / elapsed / 1e12, 3),
            "max_memory_allocated_gb": round(torch.cuda.max_memory_allocated(device) / 1024**3, 3),
            "sample_value": float(c[0, 0].detach().cpu()),
        }
        del a, b, c
        torch.cuda.empty_cache()
    else:
        gpu_smoke = {"ran": False, "reason": "torch.cuda.is_available() is False"}
except Exception as exc:
    gpu_smoke = {"ran": False, "error": repr(exc)}

report["gpu_smoke_test"] = gpu_smoke
print(json.dumps(gpu_smoke, indent=2))

{
  "ran": true,
  "device": "NVIDIA A100-SXM4-80GB",
  "dtype": "torch.float16",
  "matrix_size": 4096,
  "repeats": 8,
  "elapsed_seconds": 0.5518286842852831,
  "approx_tflops": 1.992,
  "max_memory_allocated_gb": 0.133,
  "sample_value": -74.0625
}


## 7. PyTorch Geometric smoke test

Several project notebooks depend on PyTorch Geometric. This creates a tiny graph batch and runs one graph convolution forward/backward pass on GPU if CUDA is available.

In [8]:
pyg_smoke = {"import_ok": False, "ran": False}

try:
    import torch
    import torch_geometric
    from torch_geometric.data import Data, Batch
    from torch_geometric.nn import GCNConv, global_mean_pool

    pyg_smoke.update({
        "import_ok": True,
        "torch_geometric_version": torch_geometric.__version__,
    })

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    g1 = Data(
        x=torch.randn(4, 8),
        edge_index=torch.tensor([[0, 1, 2, 2, 3, 0], [1, 0, 1, 3, 2, 3]], dtype=torch.long),
        y=torch.tensor([1.0]),
    )
    g2 = Data(
        x=torch.randn(3, 8),
        edge_index=torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long),
        y=torch.tensor([0.0]),
    )
    batch = Batch.from_data_list([g1, g2]).to(device)
    conv = GCNConv(8, 16).to(device)
    head = torch.nn.Linear(16, 1).to(device)
    out = conv(batch.x, batch.edge_index).relu()
    graph_out = global_mean_pool(out, batch.batch)
    pred = head(graph_out).view(-1)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(pred, batch.y.float())
    loss.backward()

    pyg_smoke.update({
        "ran": True,
        "device": str(device),
        "loss": float(loss.detach().cpu()),
        "prediction_shape": list(pred.shape),
    })
except Exception as exc:
    pyg_smoke["error"] = repr(exc)

report["torch_geometric_smoke_test"] = pyg_smoke
print(json.dumps(pyg_smoke, indent=2))

{
  "import_ok": false,
  "ran": false,
  "error": "ModuleNotFoundError(\"No module named 'torch_geometric'\")"
}


## 8. RDKit chemistry smoke test

In [9]:
rdkit_smoke = {"import_ok": False, "ran": False}

try:
    import rdkit
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

    smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"  # aspirin
    mol = Chem.MolFromSmiles(smiles)
    mol_h = Chem.AddHs(mol)
    embed_status = AllChem.EmbedMolecule(mol_h, randomSeed=42)
    if embed_status == 0:
        AllChem.UFFOptimizeMolecule(mol_h, maxIters=50)

    rdkit_smoke.update({
        "import_ok": True,
        "rdkit_version": rdkit.__version__,
        "ran": True,
        "smiles": smiles,
        "canonical_smiles": Chem.MolToSmiles(mol),
        "num_atoms": mol.GetNumAtoms(),
        "num_bonds": mol.GetNumBonds(),
        "molecular_weight": round(Descriptors.MolWt(mol), 4),
        "tpsa": round(rdMolDescriptors.CalcTPSA(mol), 4),
        "conformer_embedding_status": embed_status,
    })
except Exception as exc:
    rdkit_smoke["error"] = repr(exc)

report["rdkit_smoke_test"] = rdkit_smoke
print(json.dumps(rdkit_smoke, indent=2))

{
  "import_ok": false,
  "ran": false,
  "error": "ModuleNotFoundError(\"No module named 'rdkit'\")"
}


## 9. CPU, RAM, and disk snapshot

`psutil` is optional. If it is not installed, the notebook still reports CPU count and project disk usage via the Python standard library.

In [10]:
system_snapshot = {
    "cpu_count_logical": os.cpu_count(),
    "project_disk_usage": {},
}

try:
    usage = shutil.disk_usage(PROJECT_ROOT)
    system_snapshot["project_disk_usage"] = {
        "total_gb": round(usage.total / 1024**3, 3),
        "used_gb": round(usage.used / 1024**3, 3),
        "free_gb": round(usage.free / 1024**3, 3),
    }
except Exception as exc:
    system_snapshot["project_disk_usage_error"] = repr(exc)

try:
    import psutil

    vm = psutil.virtual_memory()
    system_snapshot.update({
        "psutil_version": importlib_metadata.version("psutil"),
        "cpu_count_physical": psutil.cpu_count(logical=False),
        "ram_total_gb": round(vm.total / 1024**3, 3),
        "ram_available_gb": round(vm.available / 1024**3, 3),
    })
except Exception as exc:
    system_snapshot["psutil"] = f"not available or failed: {exc!r}"

report["system_snapshot"] = system_snapshot
print(json.dumps(system_snapshot, indent=2))

{
  "cpu_count_logical": 128,
  "project_disk_usage": {
    "total_gb": 34048.0,
    "used_gb": 14285.572,
    "free_gb": 19762.428
  },
  "psutil_version": "7.2.2",
  "cpu_count_physical": 128,
  "ram_total_gb": 2015.696,
  "ram_available_gb": 1970.618
}


## 10. Save the report

In [11]:
if SAVE_REPORT:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
    report_path = REPORT_DIR / f"environment_report_{stamp}.json"
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(f"Saved report to: {report_path}")
else:
    print("SAVE_REPORT is False; report was not written.")

Saved report to: /workspace/data/environment_reports/environment_report_20260625_142340_utc.json


## How to read the results

- If `nvidia-smi` works but `torch.cuda.is_available()` is false, the NVIDIA driver is visible but the installed PyTorch build is probably CPU-only or CUDA-incompatible.
- If `torch.version.cuda` differs from `nvcc --version`, that is usually okay: PyTorch ships or links against its own CUDA runtime. The NVIDIA driver still needs to be new enough for that runtime.
- If the PyTorch Geometric smoke test fails, check that `torch_geometric` and its compiled companion packages match the installed PyTorch/CUDA version.
- If RDKit fails, the chemistry notebooks may still start, but molecule parsing, featurization, conformers, and scaffold splits will be unreliable until RDKit is fixed.
- Keep the saved JSON report with training logs; it is surprisingly useful when a model only fails on the cluster. Machines have personalities too, unfortunately.